<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Embedding Drift Detection Over Time**

Estimated time needed: **40** minutes


## Overview

Modern AI data platforms rely on embeddings for semantic retrieval, ranking, recommendation, anomaly detection, and agent memory. In production, these embeddings can drift as user behavior, source systems, or upstream feature pipelines change. If drift is not detected early, retrieval quality declines, relevance drops, and downstream LLM responses can become unreliable.

In this lab, you will build a compact drift-monitoring workflow that simulates temporal windows, generates embeddings in a shared space, computes quantitative drift signals, and converts those signals into operational thresholds and remediation actions. This reflects real-world data engineering responsibilities in MLOps and enterprise AI governance, where teams must continuously validate model inputs and representational stability.

By the end, you will have an end-to-end notebook pattern for embedding drift detection that can be adapted to vector databases, RAG pipelines, or semantic recommendation systems.


## Table of Contents

<div class="alert alert-block alert-info" style="margin-top: 20px">
  <ol>
    <li><a href="#Overview">Overview</a></li>
    <li><a href="#Materials-and-Methods">Materials and Methods</a></li>
    <li><a href="#Dataset-Requirements">Dataset Requirements</a></li>
    <li><a href="#Objectives">Objectives</a></li>
    <li><a href="#Setup,-Installations-and-Data-Checks">Setup, Installations and Data Checks</a></li>
    <li><a href="#Embedding-Construction-Across-Time-Windows">Embedding Construction Across Time Windows</a></li>
    <li><a href="#Drift-Metrics-and-Interpretation">Drift Metrics and Interpretation</a></li>
    <li><a href="#Thresholds-and-Remediation-Actions">Thresholds and Remediation Actions</a></li>
    <li><a href="#Student-Tasks">Student Tasks</a></li>
    <li><a href="#Conclusions">Conclusions</a></li>
    <li><a href="#Author">Author</a></li>
  </ol>
</div>


## Materials and Methods

This lab uses a compact embedding monitoring pipeline with the following components:

- Feature standardization using **[StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html)**.
- Low-dimensional embedding projection with **[PCA](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html)** to create a stable comparison space over time windows.
- Drift measurement using centroid-level **[Euclidean distance](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.euclidean_distances.html)** and distribution-level **[Wasserstein distance](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.wasserstein_distance.html)**.
- Rule-based alerting thresholds and remediation recommendations aligned with MLOps operations.

Architectural Trade-offs: PCA is fast and interpretable but may lose nonlinear structure compared with deep embedding models. Euclidean centroid shift is computationally lightweight for global movement monitoring, while Wasserstein distance is more descriptive for distribution shifts but can cost more at scale. In production, teams balance speed vs. fidelity and latency vs. memory constraints depending on SLA, data volume, and observability requirements.


## Dataset Requirements

To keep experimentation fast and reproducible, this lab uses the built-in **[Breast Cancer Wisconsin dataset](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html)** from `sklearn.datasets`. We treat its numeric feature vectors as a stand-in for production embedding inputs and simulate temporal windows through controlled feature shifts.

This dataset is provided under the <a href="https://opensource.org/licenses/BSD-3-Clause" target="_blank">BSD 3-Clause</a> license and is safe for commercial use.


## Objectives

After completing this lab, you will be able to:

- Analyze embedding behavior across time windows.
- Recommend thresholds and remediation actions for drift events.


## Setup, Installations and Data Checks

The first operational step in enterprise notebooks is dependency control. We explicitly install all packages needed by this lab so execution is reproducible across local, cloud, and CI notebook runners.

Run the next cell to install required libraries quietly.


In [ ]:
%pip install -q pandas numpy scikit-learn seaborn matplotlib scipy

With dependencies installed, we can now import libraries and configure deterministic behavior. This is an important reliability pattern in data engineering because nondeterministic notebooks are difficult to debug, test, and promote through environments.


### Import Libraries

The imports below include data handling, embedding projection, drift metrics, and visualization tools. Refer to official documentation for **[pandas](https://pandas.pydata.org/docs/)**, **[NumPy](https://numpy.org/doc/)**, and **[seaborn](https://seaborn.pydata.org/)** for deeper API details.


In [ ]:
# Import core tabular and numeric libraries.
import pandas as pd
import numpy as np

# Import visualization libraries for drift dashboards.
import matplotlib.pyplot as plt
import seaborn as sns

# Import dataset and preprocessing utilities.
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Import statistical distance for distribution drift measurement.
from scipy.stats import wasserstein_distance

# Keep output clean and ensure repeatable randomness.
import warnings
warnings.filterwarnings("ignore")
np.random.seed(42)
sns.set_theme(style="whitegrid")

The environment is now ready. Next, we load the dataset and validate its baseline quality before any modeling step. Data quality checks should happen early so failed assumptions are caught before expensive downstream computations.


### Load Dataset

We load the built-in dataset directly from scikit-learn and create a working DataFrame. In enterprise pipelines, this stage typically corresponds to reading from a feature store, warehouse, or data lake snapshot.


In [ ]:
# Load a built-in dataset from scikit-learn as a DataFrame.
dataset = load_breast_cancer(as_frame=True)

# Copy data locally and standardize the target column name.
df = dataset.frame.copy()
df = df.rename(columns={"target": "label"})

# Display shape and sample rows to verify ingestion.
print("Dataset shape:", df.shape)
df.head()

The output should confirm a fully numeric feature matrix plus a binary label. Before generating embeddings, we need governance-style quality checks: schema validation, null checks, duplicate checks, and domain checks on target values.


### Data Quality Checks

These checks mirror CI/CD gates commonly used in production data platforms. If any check fails, the pipeline should block drift scoring and trigger data contract investigation.


In [ ]:
# Validate schema against expected fields from sklearn metadata.
expected_columns = list(dataset.feature_names) + ["label"]
schema_is_valid = list(df.columns) == expected_columns

# Check common enterprise quality rules.
null_cell_count = int(df.isnull().sum().sum())
duplicate_row_count = int(df.duplicated().sum())
label_values = sorted(df["label"].unique().tolist())

# Print a compact quality report for CI/CD style checks.
print("Schema valid:", schema_is_valid)
print("Null cells:", null_cell_count)
print("Duplicate rows:", duplicate_row_count)
print("Label values:", label_values)

A healthy quality report gives us a trustworthy base for temporal simulation. Next, we create multiple time windows with progressive feature perturbations to emulate embedding drift in production.


## Embedding Construction Across Time Windows

In real systems, drift appears when feature distributions shift over days or weeks. To replicate that behavior, we define a small transformation function that applies scaling, bias, and random noise. This gives us controlled drift intensity while preserving interpretability.


In [ ]:
# Define a helper to simulate temporal feature shift.
def apply_window_shift(frame, scale=1.0, bias=0.0, noise=0.0):
    shifted = frame.copy()
    feature_cols = shifted.columns.drop("label")

    # Apply deterministic scaling and bias shifts.
    shifted.loc[:, feature_cols] = shifted[feature_cols] * scale + bias

    # Inject random noise to mimic unstable production signals.
    if noise > 0:
        shifted.loc[:, feature_cols] += np.random.normal(
            loc=0.0, scale=noise, size=shifted[feature_cols].shape
        )

    return shifted

The helper defines our synthetic temporal behavior. We now generate four windows, from baseline to high drift, and combine them into one analysis table that represents a time series of embedding inputs.


In [ ]:
# Sample a stable baseline subset to keep windows comparable.
base_df = df.sample(n=420, random_state=42).reset_index(drop=True)

# Define progressive temporal windows with increasing drift intensity.
window_specs = [
    ("W1_baseline", 1.00, 0.00, 0.00),
    ("W2_light", 1.01, 0.02, 0.01),
    ("W3_moderate", 1.03, 0.06, 0.02),
    ("W4_high", 1.08, 0.12, 0.03),
]

# Materialize one synthetic snapshot per time window.
window_frames = []
for window_id, scale, bias, noise in window_specs:
    shifted = apply_window_shift(base_df, scale=scale, bias=bias, noise=noise)
    shifted["window_id"] = window_id
    window_frames.append(shifted)

emb_input = pd.concat(window_frames, ignore_index=True)
print("Combined temporal dataset shape:", emb_input.shape)

We now have a temporal dataset where each record belongs to a specific window. Next, we project all windows into one shared embedding coordinate system so distances are directly comparable across time.


In [ ]:
# Select feature columns and isolate the baseline window.
feature_cols = emb_input.columns.drop(["label", "window_id"])
baseline_mask = emb_input["window_id"] == "W1_baseline"

# Fit scaler and PCA only on baseline for a stable reference space.
scaler = StandardScaler()
pca = PCA(n_components=2, random_state=42)
baseline_scaled = scaler.fit_transform(emb_input.loc[baseline_mask, feature_cols])
pca.fit(baseline_scaled)

# Transform all windows into the same embedding coordinate system.
all_scaled = scaler.transform(emb_input[feature_cols])
all_embeddings = pca.transform(all_scaled)
emb_input["emb_x"] = all_embeddings[:, 0]
emb_input["emb_y"] = all_embeddings[:, 1]
emb_input[["window_id", "emb_x", "emb_y"]].head()

At this point, each row has a 2D embedding coordinate. We can now quantify how far each window moves away from the baseline in both direction and distribution shape.


## Drift Metrics and Interpretation

A robust drift signal usually combines multiple views:

- **Centroid Euclidean distance** for geometric movement of the embedding cloud center.
- **Wasserstein distance** for distributional displacement in each embedding dimension.

Using both helps reduce blind spots that appear when relying on only one metric.


In [ ]:
# Cache baseline and define ordered windows.
window_order = sorted(emb_input["window_id"].unique())
baseline_vecs = emb_input.loc[
    emb_input["window_id"] == "W1_baseline", ["emb_x", "emb_y"]
].to_numpy()

# Measure centroid movement using Euclidean distance.
centroid_rows = []
base_centroid = baseline_vecs.mean(axis=0)
for idx, window_id in enumerate(window_order, start=1):
    current_vecs = emb_input.loc[
        emb_input["window_id"] == window_id, ["emb_x", "emb_y"]
    ].to_numpy()
    curr_centroid = current_vecs.mean(axis=0)
    centroid_l2 = float(np.linalg.norm(curr_centroid - base_centroid))
    centroid_rows.append((idx, window_id, centroid_l2))

centroid_df = pd.DataFrame(centroid_rows, columns=["window_rank", "window_id", "centroid_l2"])
centroid_df

The centroid metric captures global direction changes, but it does not fully describe spread or shape shifts. We now add Wasserstein distance to detect broader distribution movement.


In [ ]:
# Measure distribution shift with Wasserstein distance on each axis.
wasserstein_rows = []
for window_id in window_order:
    current_vecs = emb_input.loc[
        emb_input["window_id"] == window_id, ["emb_x", "emb_y"]
    ].to_numpy()

    # Average x-axis and y-axis Wasserstein values for one score.
    w_x = wasserstein_distance(baseline_vecs[:, 0], current_vecs[:, 0])
    w_y = wasserstein_distance(baseline_vecs[:, 1], current_vecs[:, 1])
    wasserstein_rows.append((window_id, float((w_x + w_y) / 2.0)))

wasserstein_df = pd.DataFrame(wasserstein_rows, columns=["window_id", "mean_wasserstein"])
wasserstein_df

Now we fuse the two metric families into one operational score. Weighted scoring is common in monitoring systems because it allows teams to tune sensitivity to their domain risk profile.


In [ ]:
# Merge metric views into a single drift table.
drift_df = centroid_df.merge(wasserstein_df, on="window_id", how="left")

# Create a composite score balancing direction and spread.
drift_df["drift_score"] = 0.6 * drift_df["centroid_l2"] + 0.4 * drift_df["mean_wasserstein"]

# Keep rows in chronological order for reporting.
drift_df = drift_df.sort_values("window_rank").reset_index(drop=True)
drift_df.round(6)

The resulting table should show low scores near baseline and progressively higher values as synthetic drift increases. Next, we convert these values into warning and critical states that can drive automated actions.


## Thresholds and Remediation Actions

Threshold design is a governance decision, not just a statistical one. A common starting policy is mean plus k times standard deviation on stable reference windows. This creates simple, explainable limits that operations teams can audit and refine.


In [ ]:
# Use early stable windows to derive baseline behavior.
reference = drift_df.loc[drift_df["window_rank"] <= 2, "drift_score"]

# Build warning and critical limits from mean and standard deviation.
mu = float(reference.mean())
sigma = float(reference.std(ddof=0))
warn_threshold = mu + (2 * sigma)
critical_threshold = mu + (3 * sigma)

# Print thresholds for governance and alert policy reviews.
print("Warning threshold:", round(warn_threshold, 6))
print("Critical threshold:", round(critical_threshold, 6))

With thresholds defined, we map each window to a drift level. In mature systems, this output often feeds alert routers, incident tickets, or retraining orchestration workflows.


In [ ]:
# Classify each window using the threshold policy.
def classify_drift(score, warn, critical):
    if score >= critical:
        return "critical"
    if score >= warn:
        return "warning"
    return "normal"

# Apply classification to every time window.
drift_df["drift_level"] = drift_df["drift_score"].apply(
    lambda x: classify_drift(x, warn_threshold, critical_threshold)
)
drift_df

This table is your core monitoring artifact: one row per window with a quantitative score and an interpretable severity label. We now visualize score trajectories so drift acceleration is easier to spot during operations reviews.


### Visualize Drift Timeline

Trend charts are essential for operational observability. They help teams detect not only threshold crossing but also persistent upward movement that may justify proactive intervention.


In [ ]:
# Plot drift score trends and policy thresholds over time.
plt.figure(figsize=(10, 5))
plt.plot(drift_df["window_id"], drift_df["drift_score"], marker="o", linewidth=2, label="Drift score")

# Overlay warning and critical boundaries for operations teams.
plt.axhline(warn_threshold, color="orange", linestyle="--", label="Warning threshold")
plt.axhline(critical_threshold, color="red", linestyle="--", label="Critical threshold")

# Finalize chart styling for readability in runbooks.
plt.title("Embedding Drift Score by Time Window")
plt.xlabel("Time window")
plt.ylabel("Composite drift score")
plt.legend()
plt.tight_layout()
plt.show()

Interpretation: if later windows approach or cross warning and critical lines, embeddings are no longer behaviorally stable. This should trigger data quality triage, feature drift diagnostics, and potentially model refresh procedures.


### Visualize Embedding Shift in 2D

A geometric view of baseline versus high-drift windows helps analysts explain why metric values changed. Even if labels stay balanced, representation geometry may still drift and degrade retrieval quality.


In [ ]:
# Compare baseline and latest window embeddings side by side.
plot_df = emb_input[emb_input["window_id"].isin(["W1_baseline", "W4_high"])].copy()

# Use seaborn for a clean two-color embedding map.
plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=plot_df,
    x="emb_x",
    y="emb_y",
    hue="window_id",
    alpha=0.65,
)

# Add labels and rendering options for interpretability.
plt.title("Embedding Space: Baseline vs High Drift Window")
plt.xlabel("Embedding component 1")
plt.ylabel("Embedding component 2")
plt.tight_layout()
plt.show()

When point clouds separate noticeably, retrieval neighborhoods can shift and produce different nearest neighbors for similar queries. This is a concrete signal that semantic systems may need recalibration.


### Recommend Remediation Actions

Monitoring is only useful if it informs action. We now map drift levels to operational responses. This aligns with incident management practices where each severity tier has a predefined runbook.


In [ ]:
# Map drift levels to operational remediation actions.
def remediation_action(level):
    if level == "critical":
        return "Trigger retraining, freeze risky deployment, and run root-cause analysis."
    if level == "warning":
        return "Increase monitoring cadence and run targeted feature diagnostics."
    return "Continue standard monitoring and weekly validation checks."

# Attach actions and show an execution-ready response table.
drift_df["recommended_action"] = drift_df["drift_level"].apply(remediation_action)
drift_df[["window_id", "drift_score", "drift_level", "recommended_action"]]

This response table can be exported to alert systems, dashboards, or governance reports. Architectural Trade-offs appear again here: tighter thresholds catch issues earlier but increase false positives, while looser thresholds reduce alert fatigue but risk delayed intervention.


## Student Tasks

Complete the following hands-on tasks to strengthen your threshold design and remediation strategy decisions.


<div class="alert alert-danger alertdanger" style="margin-top: 20px">
<h1> Question #1: </h1>
<b>Recompute the composite drift score using equal weights (0.5 and 0.5) for centroid cosine and Wasserstein metrics. Compare the ranking of windows before and after the change.</b>
</div>


In [ ]:
# TODO: Copy drift_df and create a new score with equal weights.
# TODO: Add the new score as a column named drift_score_equal_weight.
# TODO: Sort by the new score and compare the order with the original.

student_df = drift_df.copy()

# TODO: Replace the placeholder output with your final comparison table.
student_df[["window_id", "drift_score"]]


<details><summary>Click here for the solution</summary>

```python
# Create a working copy to avoid mutating the original table.
student_df = drift_df.copy()

# Compute alternative score with equal weights.
student_df["drift_score_equal_weight"] = (
    0.5 * student_df["centroid_l2"] + 0.5 * student_df["mean_wasserstein"]
)

# Compare ranking under the original and new scoring policy.
original_rank = student_df.sort_values("drift_score", ascending=False)[["window_id", "drift_score"]]
new_rank = student_df.sort_values("drift_score_equal_weight", ascending=False)[["window_id", "drift_score_equal_weight"]]

print("Original ranking by drift_score:")
display(original_rank.reset_index(drop=True))

print("New ranking by drift_score_equal_weight:")
display(new_rank.reset_index(drop=True))
```

</details>


<div class="alert alert-danger alertdanger" style="margin-top: 20px">
<h1> Question #2: </h1>
<b>Design a stricter alert policy by reducing the warning and critical multipliers from (2, 3) to (1.5, 2.5). Reclassify drift levels and summarize how many windows fall into each level.</b>
</div>


In [ ]:
# TODO: Define stricter multipliers for warning and critical thresholds.
# TODO: Recompute thresholds using the same reference windows.
# TODO: Reclassify each window and count level frequencies.

strict_warn_k = 1.5
strict_critical_k = 2.5

# TODO: Replace placeholder output with your classification summary.
drift_df[["window_id", "drift_level"]]


<details><summary>Click here for the solution</summary>

```python
# Reuse stable reference windows for threshold estimation.
reference = drift_df.loc[drift_df["window_rank"] <= 2, "drift_score"]
mu = float(reference.mean())
sigma = float(reference.std(ddof=0))

# Apply stricter multipliers than the base policy.
strict_warn_threshold = mu + (1.5 * sigma)
strict_critical_threshold = mu + (2.5 * sigma)

# Create a copy and classify with strict thresholds.
strict_df = drift_df.copy()
strict_df["strict_level"] = strict_df["drift_score"].apply(
    lambda x: classify_drift(x, strict_warn_threshold, strict_critical_threshold)
)

print("Strict warning threshold:", round(strict_warn_threshold, 6))
print("Strict critical threshold:", round(strict_critical_threshold, 6))

display(strict_df[["window_id", "drift_score", "strict_level"]])
display(strict_df["strict_level"].value_counts().rename_axis("strict_level").to_frame("window_count"))
```

</details>


## Conclusions

You implemented an end-to-end embedding drift workflow that:

- Created temporal windows with controlled distribution changes.
- Built a shared embedding space for cross-window comparison.
- Quantified drift using directional and distribution metrics.
- Converted drift scores into actionable operational severity levels.

This notebook structure can be extended to real vector stores and production telemetry streams by replacing synthetic windows with timestamped embedding logs.


## Author

<a href="https://author.skills.network/instructors/dmytro_shliakhovskyi">Dmytro Shliakhovskyi</a>

## Change Log
| Date (YYYY-MM-DD) | Version | Changed By | Change Description |
| ----------------- | ------- | ---------- | ------------------ |
| 2026-04-23 | 01 | Dmytro Shliakhovskyi | Lab created |

<h3 align="center"> © IBM Corporation 2020. All rights reserved. <h3/>
